# Fast No RL 04 - Held-Out Evaluation and Selection

Freeze the selected variant, then measure it on the held-out suite. Fast No RL completion files are read. This records a selection, not an assertion that the agent is strong or accepted by the platform.

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project Setup

Uses `configs/fast_no_rl.yaml`. No league, self-play, PPO, or DQN is run. Colab's installed PyTorch is preserved.

In [ ]:
from pathlib import Path
import sys
import subprocess
from IPython.display import display
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

def show_figure(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl/non_rl.py").is_file():
    raise FileNotFoundError(f"Place the updated project contents directly in {PROJECT_ROOT}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                       str(PROJECT_ROOT / "requirements_colab.txt")])
sys.path.insert(0, str(PROJECT_ROOT))
from chess_rl.non_rl import load_no_rl_config
from chess_rl.reproducibility import read_json, sha256
cfg = load_no_rl_config(PROJECT_ROOT, "fast_no_rl.yaml")
print("Run:", cfg["run_id"], "| Variant:", cfg["non_rl"]["variant"])
print("Root:", PROJECT_ROOT)


## Confirm the Candidate

Complete development experiments before this step. The lock covers source hashes, search settings, and any supervised checkpoint. Do not reuse this held-out suite as fresh evidence after tuning to its results.

In [ ]:
from chess_rl.non_rl import candidate_spec
spec = candidate_spec(PROJECT_ROOT, cfg)
print("Variant:", spec["variant"])
print("Checkpoint:", spec["checkpoint"] or "None: classical, no training")
print("Held-out games per opponent:", cfg["evaluation"]["heldout_games"])

## Run or Resume Held-Out Games

The candidate is locked before games begin. All opponents use the same reserved fixtures and clocks. The selected run may be exported even if its playing strength is poor, but final runtime checks can still fail.

In [ ]:
from chess_rl.non_rl import evaluate_and_freeze_no_rl
selection = evaluate_and_freeze_no_rl(PROJECT_ROOT, cfg)
print("Selection:", PROJECT_ROOT / "results" / cfg["run_id"] / "no_rl/final_selection.json")
print("Training status:", selection["training_status"])

## Read the Results

These are observed scores, not guaranteed future performance. Colab does not reproduce the platform's CPU speed, isolation, or hidden openings. Final packaging/compliance is exclusively in no-RL notebook 05.

In [ ]:
from chess_rl.plots import plot_matches
show_figure(plot_matches(PROJECT_ROOT, cfg["run_id"], selection["heldout"], "no_rl_heldout"))